# Legal ReAct v2 Single-Case Comparison

This notebook compares the original `legal_react` baseline against `legal_react_v2` on one known working case.

Case selector:
- year = 2022
- docket_id = 63356973
- model_name = openai:gpt-5
- max_agent_steps = 60

The goal is to trace both runs with MLflow, compare the predicted offense level against the expected offense level from the strict table, and surface the first concrete mismatch or failure mode for iteration.

In [ ]:
from __future__ import annotations

import importlib
import json
from pprint import pprint

import mlflow
import mlflow.langchain

import baselines.legal_react as legal_react
import baselines.legal_react.agent as legal_react_agent
import baselines.legal_react.single_case as legal_react_single_case
import baselines.legal_react_v2 as legal_react_v2
import baselines.legal_react_v2.agent as legal_react_v2_agent
import baselines.legal_react_v2.single_case as legal_react_v2_single_case

from baselines.legal_rag.runtime import resolve_model_name, resolve_spark_session, resolve_strict_table

legal_react_agent = importlib.reload(legal_react_agent)
legal_react_single_case = importlib.reload(legal_react_single_case)
legal_react = importlib.reload(legal_react)
legal_react_v2_agent = importlib.reload(legal_react_v2_agent)
legal_react_v2_single_case = importlib.reload(legal_react_v2_single_case)
legal_react_v2 = importlib.reload(legal_react_v2)

orig_load_config = legal_react.load_config
orig_run_single_case_prediction = legal_react.run_single_case_prediction
orig_score_prediction = legal_react.score_prediction
v2_load_config = legal_react_v2.load_config
v2_run_single_case_prediction = legal_react_v2.run_single_case_prediction
v2_score_prediction = legal_react_v2.score_prediction
fetch_case_record = legal_react_v2.fetch_case_record
render_system_prompt = legal_react_v2.render_system_prompt

In [ ]:
YEAR = 2022
DOCKET_ID = 63356973
GOVERNMENT_SM_DOC_ID = None
MODEL_NAME = resolve_model_name()
MAX_AGENT_STEPS = 60
STRICT_TABLE = resolve_strict_table()
MLFLOW_EXPERIMENT = "/Shared/legal_react_v2_single_case"

orig_config = orig_load_config()
v2_config = v2_load_config()

{
    "year": YEAR,
    "docket_id": DOCKET_ID,
    "government_sm_doc_id": GOVERNMENT_SM_DOC_ID,
    "model_name": MODEL_NAME,
    "max_agent_steps": MAX_AGENT_STEPS,
    "strict_table": STRICT_TABLE,
    "mlflow_experiment": MLFLOW_EXPERIMENT,
    "original_execution_env": orig_config.execution_env,
    "v2_execution_env": v2_config.execution_env,
}

In [ ]:
spark = resolve_spark_session(app_name="legal-react-v2-single-case")
case_record = fetch_case_record(
    spark=spark,
    table_name=STRICT_TABLE,
    docket_id=DOCKET_ID,
    year=YEAR,
    government_sm_doc_id=GOVERNMENT_SM_DOC_ID,
)

{
    "year": case_record["year"],
    "docket_id": case_record["docket_id"],
    "government_sm_doc_id": case_record["government_sm_doc_id"],
    "expected_offense_level_total": case_record["expected_offense_level_total"],
    "expected_criminal_history_category": case_record["expected_criminal_history_category"],
    "expected_guidelines_low_months": case_record["expected_guidelines_low_months"],
    "expected_guidelines_high_months": case_record["expected_guidelines_high_months"],
    "charges_or_offense": case_record["charges_or_offense"],
    "case_summary_preview": case_record["case_summary"][:3000],
}

In [ ]:
if mlflow.active_run() is not None:
    mlflow.end_run()

mlflow.set_experiment(MLFLOW_EXPERIMENT)
mlflow.langchain.autolog(log_traces=True, silent=True)

{
    "tracking_uri": mlflow.get_tracking_uri(),
    "experiment": MLFLOW_EXPERIMENT,
}

In [ ]:
# with mlflow.start_run(run_name=f"legal_react_original_{YEAR}_{DOCKET_ID}") as run:
#     mlflow.log_params(
#         {
#             "baseline": "legal_react",
#             "year": YEAR,
#             "docket_id": DOCKET_ID,
#             "government_sm_doc_id": case_record.get("government_sm_doc_id"),
#             "model_name": MODEL_NAME,
#             "max_agent_steps": MAX_AGENT_STEPS,
#         }
#     )
#     original_result = orig_run_single_case_prediction(
#         config=orig_config,
#         summary_text=case_record["case_summary"],
#         model_name=MODEL_NAME,
#         year=YEAR,
#         max_agent_steps=MAX_AGENT_STEPS,
#     )
#     original_metrics = orig_score_prediction(original_result["prediction"], case_record)
#     mlflow.log_dict(original_result["prediction"], "original/prediction.json")
#     mlflow.log_dict({"messages": original_result["messages"]}, "original/messages.json")
#     mlflow.log_dict(original_metrics, "original/metrics.json")
#     original_run_id = run.info.run_id

# {
#     "mlflow_run_id": original_run_id,
#     "prediction": original_result["prediction"],
#     "metrics": original_metrics,
# }

In [ ]:
with mlflow.start_run(run_name=f"legal_react_v2_{YEAR}_{DOCKET_ID}") as run:
    mlflow.log_params(
        {
            "baseline": "legal_react_v2",
            "year": YEAR,
            "docket_id": DOCKET_ID,
            "government_sm_doc_id": case_record.get("government_sm_doc_id"),
            "model_name": MODEL_NAME,
            "max_agent_steps": MAX_AGENT_STEPS,
        }
    )
    v2_result = v2_run_single_case_prediction(
        config=v2_config,
        summary_text=case_record["case_summary"],
        model_name=MODEL_NAME,
        year=YEAR,
        max_agent_steps=MAX_AGENT_STEPS,
    )
    v2_metrics = v2_score_prediction(v2_result["prediction"], case_record)
    mlflow.log_dict(v2_result["prediction"], "v2/prediction.json")
    mlflow.log_dict({"messages": v2_result["messages"]}, "v2/messages.json")
    mlflow.log_dict(v2_metrics, "v2/metrics.json")
    v2_run_id = run.info.run_id

{
    "mlflow_run_id": v2_run_id,
    "prediction": v2_result["prediction"],
    "metrics": v2_metrics,
}

In [ ]:
comparison = {
    "expected_offense_level_total": case_record.get("expected_offense_level_total"),
    "original_predicted_offense_level_total": original_result["prediction"].get("predicted_offense_level_total"),
    "v2_predicted_offense_level_total": None if v2_result["prediction"].get("offense_level") is None else str(v2_result["prediction"].get("offense_level")),
    "original_exact_match": original_metrics.get("offense_level_total_exact_match"),
    "v2_exact_match": v2_metrics.get("offense_level_total_exact_match"),
    "original_rationale": original_result["prediction"].get("rationale"),
    "v2_justifications": v2_result["prediction"].get("justifications"),
}
comparison

In [ ]:
{
    "original_final_message_text": original_result["final_message_text"][:4000],
    "v2_final_message_text": v2_result["final_message_text"][:4000],
}